# 4. Weather and generation

Build renewable features and a first generation forecast. This notebook
adds ERA5 weather data, computes the **clear-sky index** to isolate
cloud effects from the sun's position, and builds baseline generation
forecasts for wind and solar.

## Objectives

- Fetch ERA5 reanalysis data (or use stubs if no CDS account).
- Inspect the raw weather data: grid cells, variables, temporal resolution.
- Compute clear-sky GHI using pvlib and derive the clear-sky index.
- Analyse wind speed patterns and their relationship to price.
- Build a baseline generation forecast (persistence).
- Merge weather and generation features onto the price dataset.
- Implement `data.load_era5()` and `features.clear_sky_index()`.

## Prerequisites

- Notebooks 01–03 completed.
- ERA5 CDS account (optional — stubs provided if credentials are missing).
- `pvlib` installed for clear-sky modelling.

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grian.config import load_config, repo_root
from grian.features import clear_sky_index
from grian.viz import apply_style, save_fig

cfg = load_config()
apply_style()
warnings.filterwarnings("ignore", category=FutureWarning)

REGION = cfg["region"]
START = cfg["train_start"]
END = cfg["test_end"]

REGION_COLORS = {
    "NSW1": "#2196F3", "QLD1": "#FF9800", "VIC1": "#4CAF50",
    "SA1": "#F44336", "TAS1": "#9C27B0",
}

# Representative location for SA1 (Adelaide)
SA1_LAT, SA1_LON = -34.93, 138.60

---
## 1. ERA5 weather data

ERA5 is a global weather reanalysis from the Copernicus Climate Data
Store. It provides hourly data on a 0.25° grid, including:

- `ssrd` — Surface solar radiation downwards (J/m², cumulative per hour)
- `t2m` — 2-metre temperature (K)
- `u100`, `v100` — 100-metre wind components (m/s)

If you don't have CDS credentials, we create a stub DataFrame with
the right schema so downstream code works.

In [ ]:
from grian.data import load_era5

# Use a shorter period for weather — ERA5 downloads are slow
weather_start = "2023-01-01"
weather_end = "2024-06-30"

try:
    weather = load_era5(REGION, weather_start, weather_end,
                        cache=str(Path(cfg["paths"]["raw"]) / "era5"))
    has_era5 = True
    print(f"ERA5 loaded: {weather.shape}")
    print(f"Columns: {list(weather.columns)}")
    print(f"Index: {weather.index[0]} to {weather.index[-1]}")
except (FileNotFoundError, Exception) as e:
    print(f"ERA5 not available: {e}")
    print("\nCreating stub weather DataFrame for downstream code.")
    has_era5 = False
    # Stub with realistic synthetic data
    idx = pd.date_range(weather_start, weather_end, freq="1h", name="timestamp")
    rng = np.random.default_rng(cfg["seed"])
    hour = idx.hour
    doy = idx.dayofyear
    solar_peak = np.maximum(0, np.sin(np.pi * (hour - 6) / 12)) * \
                 (800 + 200 * np.cos(2 * np.pi * (doy - 355) / 365))
    solar_peak *= (1 + 0.1 * rng.standard_normal(len(idx)))
    solar_peak = np.maximum(0, solar_peak)
    weather = pd.DataFrame({
        "ssrd": solar_peak * 3600,
        "t2m": 288 + 10 * np.cos(2 * np.pi * (doy - 30) / 365) + \
               2 * rng.standard_normal(len(idx)),
        "u100": 5 + 3 * rng.standard_normal(len(idx)),
        "v100": 2 + 3 * rng.standard_normal(len(idx)),
    }, index=idx)
    print(f"Stub created: {weather.shape}")

In [ ]:
weather.head(12)

In [ ]:
weather.describe()

---
## 2. Map: ERA5 grid and the NEM

ERA5 data covers a grid of cells. For each NEM region, we average
across the grid cells that fall within (or near) the region boundary.

In [ ]:
regions_gdf = gpd.read_file(repo_root() / "data" / "nem_regions.geojson")

sa1_bounds = {"lat": (-38.0, -26.0), "lon": (129.0, 141.0)}
grid_lats = np.arange(sa1_bounds["lat"][0], sa1_bounds["lat"][1] + 0.25, 0.25)
grid_lons = np.arange(sa1_bounds["lon"][0], sa1_bounds["lon"][1] + 0.25, 0.25)
grid_lat_mesh, grid_lon_mesh = np.meshgrid(grid_lats, grid_lons)

fig, ax = plt.subplots(figsize=(10, 10))
regions_gdf.plot(ax=ax, color="#f0f0f0", edgecolor="white", linewidth=1.5)
regions_gdf[regions_gdf["nem_region"] == REGION].plot(
    ax=ax, color=REGION_COLORS[REGION], alpha=0.3, edgecolor=REGION_COLORS[REGION])
ax.scatter(grid_lon_mesh.flatten(), grid_lat_mesh.flatten(),
           s=5, c="black", alpha=0.4, label="ERA5 grid points")
ax.scatter([SA1_LON], [SA1_LAT], s=100, c="red", marker="*",
           zorder=5, label="Adelaide (reference)")
ax.set_xlim(125, 145)
ax.set_ylim(-40, -24)
ax.legend()
ax.set_title(f"ERA5 grid over {REGION}")
save_fig(fig, "04_era5_grid_map")
plt.show()

---
## 3. Clear-sky model and clear-sky index

The raw solar irradiance (ssrd) conflates two things:
1. The sun's position (time of day, season) — perfectly predictable.
2. Cloud cover — the thing we actually need to forecast.

The **clear-sky index** = actual GHI / clear-sky GHI isolates the
cloud effect. A value of 1.0 means perfectly clear; 0.3 means heavy
cloud. This is implemented in `grian.features.clear_sky_index()`.

In [ ]:
import pvlib

location = pvlib.location.Location(SA1_LAT, SA1_LON, tz="Australia/Adelaide")
times = weather.index.tz_localize("UTC")
clearsky = location.get_clearsky(times, model="ineichen")

actual_ghi = weather["ssrd"] / 3600  # J/m² per hour → W/m²

csi = clear_sky_index(actual_ghi, clearsky["ghi"].values)

print("Clear-sky index stats:")
print(csi.describe())

In [ ]:
sample_week = slice("2024-01-15", "2024-01-22")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(actual_ghi.loc[sample_week].index, actual_ghi.loc[sample_week].values,
         linewidth=1, label="Actual GHI", alpha=0.8)
ax1.plot(clearsky["ghi"].loc[sample_week].index,
         clearsky["ghi"].loc[sample_week].values,
         linewidth=1, linestyle="--", label="Clear-sky GHI", color="C1")
ax1.set_ylabel("GHI (W/m²)")
ax1.set_title("Clear-sky vs actual irradiance")
ax1.legend()

ax2.plot(csi.loc[sample_week].index, csi.loc[sample_week].values,
         linewidth=1, color="C2")
ax2.axhline(1.0, color="gray", linewidth=0.5, linestyle="--")
ax2.set_ylabel("Clear-sky index")
ax2.set_title("Clear-sky index (1.0 = perfectly clear)")
ax2.set_ylim(0, 1.6)

fig.tight_layout()
save_fig(fig, "04_clearsky_vs_actual")
plt.show()

---
## 4. Wind speed at 100m

Wind turbine hub height is typically 80–120m. ERA5 provides 100m wind
components (u100, v100). Wind speed = sqrt(u² + v²).

In [ ]:
weather["wind_speed_100m"] = np.sqrt(weather["u100"]**2 + weather["v100"]**2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(weather["wind_speed_100m"].dropna(), bins=100, edgecolor="none", alpha=0.7)
ax1.set_xlabel("Wind speed at 100m (m/s)")
ax1.set_ylabel("Count")
ax1.set_title(f"{REGION} — wind speed distribution")

weather["hour"] = weather.index.hour
hourly_wind = weather.groupby("hour")["wind_speed_100m"].mean()

ax2.bar(hourly_wind.index, hourly_wind.values, alpha=0.7)
ax2.set_xlabel("Hour of day")
ax2.set_ylabel("Mean wind speed (m/s)")
ax2.set_title("Diurnal wind pattern")

fig.tight_layout()
save_fig(fig, "04_wind_speed")
plt.show()

print(f"Mean wind speed: {weather['wind_speed_100m'].mean():.1f} m/s")
print(f"Calm hours (<3 m/s): {(weather['wind_speed_100m'] < 3).mean():.1%}")

---
## 5. Wind resource map across the NEM

In [ ]:
region_wind = {}
for r in ["NSW1", "QLD1", "VIC1", "SA1", "TAS1"]:
    try:
        w = load_era5(r, weather_start, weather_end,
                      cache=str(Path(cfg["paths"]["raw"]) / "era5"))
        region_wind[r] = np.sqrt(w["u100"]**2 + w["v100"]**2).mean()
    except Exception:
        approx = {"NSW1": 6.5, "QLD1": 5.8, "VIC1": 7.2, "SA1": 7.8, "TAS1": 8.5}
        region_wind[r] = approx[r]

wind_df = pd.DataFrame({"nem_region": list(region_wind.keys()),
                         "mean_wind": list(region_wind.values())})
regions_plot = regions_gdf.merge(wind_df, on="nem_region")

fig, ax = plt.subplots(figsize=(9, 11))
regions_plot.plot(ax=ax, column="mean_wind", cmap="YlGnBu",
                  edgecolor="white", linewidth=1.5, legend=True,
                  legend_kwds={"label": "Mean 100m wind speed (m/s)", "shrink": 0.5})
for _, row in regions_plot.iterrows():
    ax.annotate(f"{row['nem_region']}\n{row['mean_wind']:.1f} m/s",
                xy=(row["label_lon"], row["label_lat"]), ha="center", fontsize=9,
                fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax.set_xlim(112, 155)
ax.set_ylim(-45, -10)
ax.set_title("NEM regions — mean wind resource")
save_fig(fig, "04_wind_resource_map")
plt.show()

---
## 6. Baseline generation forecast: persistence

The simplest generation forecast: tomorrow's wind/solar output =
today's. We use SCADA data from notebook 03 to evaluate this.

In [ ]:
from nemosis import dynamic_data_compiler, static_table

cache = str(Path(cfg["nemosis_cache"]))

sample_start = "2024-01-01"
sample_end = "2024-03-31"

start_fmt = pd.Timestamp(sample_start).strftime("%Y/%m/%d %H:%M:%S")
end_fmt = pd.Timestamp(sample_end).strftime("%Y/%m/%d %H:%M:%S")

rego = static_table("Generators and Scheduled Loads", cache)
fuel_col = [c for c in rego.columns if "fuel" in c.lower()][0]
region_col = [c for c in rego.columns if "region" in c.lower()][0]

fuel_map = {
    "Natural Gas / Fuel Oil": "Gas", "Natural Gas": "Gas",
    "Water": "Hydro", "Wind": "Wind", "Solar": "Solar",
}
duid_fuel = rego[["DUID", fuel_col, region_col]].drop_duplicates(subset="DUID")
duid_fuel["fuel_simple"] = duid_fuel[fuel_col].map(fuel_map).fillna("Other")
duid_fuel = duid_fuel.rename(columns={region_col: "region"})

scada = dynamic_data_compiler(
    start_fmt, end_fmt, "DISPATCH_UNIT_SCADA", cache,
    select_columns=["SETTLEMENTDATE", "DUID", "SCADAVALUE"],
)
scada = scada.rename(columns={"SETTLEMENTDATE": "timestamp", "SCADAVALUE": "mw"})
scada["timestamp"] = pd.to_datetime(scada["timestamp"]) - pd.Timedelta(minutes=5)
scada = scada.merge(duid_fuel[["DUID", "fuel_simple", "region"]], on="DUID", how="left")
scada_sa1 = scada[scada["region"] == REGION]

wind_gen = scada_sa1[scada_sa1["fuel_simple"] == "Wind"].groupby("timestamp")["mw"].sum()
solar_gen = scada_sa1[scada_sa1["fuel_simple"] == "Solar"].groupby("timestamp")["mw"].sum()

print(f"Wind generation: {len(wind_gen):,} intervals")
print(f"Solar generation: {len(solar_gen):,} intervals")

In [ ]:
wind_30 = wind_gen.resample("30min").mean().dropna()
solar_30 = solar_gen.resample("30min").mean().dropna()

wind_persist = wind_30.shift(48)
solar_persist = solar_30.shift(48)

from grian.metrics import mae

common_idx = wind_30.index.intersection(wind_persist.dropna().index)
wind_mae = mae(wind_30.loc[common_idx], wind_persist.loc[common_idx])

common_idx_s = solar_30.index.intersection(solar_persist.dropna().index)
solar_mae = mae(solar_30.loc[common_idx_s], solar_persist.loc[common_idx_s])

print(f"Wind persistence MAE: {wind_mae:.1f} MW")
print(f"Solar persistence MAE: {solar_mae:.1f} MW")

In [ ]:
week = slice("2024-02-01", "2024-02-08")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(solar_30.loc[week].index, solar_30.loc[week].values,
         linewidth=1, label="Actual solar")
ax1.plot(solar_persist.loc[week].index, solar_persist.loc[week].values,
         linewidth=1, linestyle="--", label="Persistence (yesterday)")
ax1.set_ylabel("Solar generation (MW)")
ax1.set_title(f"{REGION} — solar: actual vs persistence")
ax1.legend()

ax2.plot(wind_30.loc[week].index, wind_30.loc[week].values,
         linewidth=1, label="Actual wind", color="C2")
ax2.plot(wind_persist.loc[week].index, wind_persist.loc[week].values,
         linewidth=1, linestyle="--", label="Persistence", color="C3")
ax2.set_ylabel("Wind generation (MW)")
ax2.set_title(f"{REGION} — wind: actual vs persistence")
ax2.legend()

fig.tight_layout()
save_fig(fig, "04_generation_persistence_forecast")
plt.show()

---
## 7. Build the weather + generation feature table

Merge the weather features and generation forecasts onto the price
dataset for use in notebook 05.

In [ ]:
processed = Path(cfg["paths"]["processed"])
df_30min = pd.read_parquet(processed / f"{REGION}_30min.parquet")

weather_30 = weather.resample("30min").mean()
weather_30["wind_speed_100m"] = np.sqrt(weather_30["u100"]**2 + weather_30["v100"]**2)
weather_30["ghi"] = weather_30["ssrd"] / 3600
weather_30["t2m_celsius"] = weather_30["t2m"] - 273.15

cs_30 = location.get_clearsky(
    weather_30.index.tz_localize("UTC"), model="ineichen")
weather_30["clearsky_ghi"] = cs_30["ghi"].values
weather_30["csi"] = clear_sky_index(
    pd.Series(weather_30["ghi"].values, index=weather_30.index),
    pd.Series(weather_30["clearsky_ghi"].values, index=weather_30.index),
)

gen_features = pd.DataFrame({
    "wind_gen": wind_30,
    "solar_gen": solar_30,
    "wind_persist": wind_persist,
    "solar_persist": solar_persist,
})

features = df_30min.join(weather_30[["wind_speed_100m", "ghi", "t2m_celsius", "csi"]],
                          how="left")
features = features.join(gen_features, how="left")

print(f"Feature table: {features.shape}")
print(f"Columns: {list(features.columns)}")
features.head()

In [ ]:
out_path = processed / f"{REGION}_30min_features.parquet"
features.to_parquet(out_path)
print(f"Saved to {out_path}")

---
## Exercises

### Exercise 1: Clear-sky index and price

What's the correlation between clear-sky index and price? Is it
linear or nonlinear?

<details><summary>Hint 1</summary>

Filter to daytime hours (8–18) when solar has a meaningful effect.
Compute Pearson and Spearman correlations between CSI and price.

</details>

<details><summary>Hint 2</summary>

Scatter-plot CSI vs price for daytime hours. Use `symlog` on the
price axis. The relationship should be negative and nonlinear.

</details>

<details><summary>Hint 3</summary>

Bin CSI into deciles and compute the median price per bin to see
the clean shape of the relationship.

</details>

<details><summary>Solution</summary>

```python
daytime = features[(features.index.hour >= 8) & (features.index.hour <= 18)].dropna(
    subset=["csi", "price"])

pearson = daytime["csi"].corr(daytime["price"])
spearman = daytime["csi"].corr(daytime["price"], method="spearman")
print(f"Daytime CSI-price Pearson: {pearson:.3f}")
print(f"Daytime CSI-price Spearman: {spearman:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(daytime["csi"], daytime["price"], s=1, alpha=0.1)
ax1.set_xlabel("Clear-sky index")
ax1.set_ylabel("Price ($/MWh)")
ax1.set_yscale("symlog", linthresh=100)
ax1.set_title(f"CSI vs price (daytime, r={pearson:.2f})")

daytime["csi_bin"] = pd.qcut(daytime["csi"], q=10, duplicates="drop")
binned = daytime.groupby("csi_bin", observed=True)["price"].median()
ax2.bar(range(len(binned)), binned.values, alpha=0.7)
ax2.set_xlabel("CSI decile (low to high)")
ax2.set_ylabel("Median price ($/MWh)")
ax2.set_title("Median price by CSI decile")

fig.tight_layout()
plt.show()
```

Higher CSI is associated with lower daytime prices. The Spearman
correlation is stronger than Pearson, confirming nonlinearity.

</details>

In [ ]:
# Your analysis here

### Exercise 2: Wind vs solar — marginal effect on price

Does wind speed or solar irradiance have a stronger marginal effect
on SA1 price?

<details><summary>Hint 1</summary>

Compute the hourly correlation of wind_speed_100m and ghi with
price. Use `features.groupby(features.index.hour)`.

</details>

<details><summary>Hint 2</summary>

Plot two lines across 24 hours. Solar should dominate during
daylight; wind may matter more at night.

</details>

<details><summary>Hint 3</summary>

SA1 has more installed wind than solar capacity, so wind's effect
may be larger overall even though solar dominates midday.

</details>

<details><summary>Solution</summary>

```python
subset = features.dropna(subset=["wind_speed_100m", "ghi", "price"])
hourly_corr = subset.groupby(subset.index.hour).apply(
    lambda g: pd.Series({
        "wind": g["wind_speed_100m"].corr(g["price"]),
        "solar": g["ghi"].corr(g["price"]),
    })
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly_corr.index, hourly_corr["wind"], "o-", label="Wind speed", color="C2")
ax.plot(hourly_corr.index, hourly_corr["solar"], "s-", label="Solar GHI", color="C1")
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Correlation with price")
ax.set_title(f"{REGION} — hourly correlation of weather with price")
ax.legend()
fig.tight_layout()
plt.show()
```

Solar has a strong negative correlation during daylight hours.
Wind's negative correlation persists through the night, and is
often larger in aggregate given SA1's wind capacity.

</details>

In [ ]:
# Your analysis here

### Exercise 3: Cloudy week vs clear week

Find a cloudy week and a clear week. Plot solar generation and price
for each. How does the price pattern differ?

<details><summary>Hint 1</summary>

Compute daily mean CSI (daytime hours only), then pick the week
with the lowest and highest mean CSI.

</details>

<details><summary>Hint 2</summary>

In the clear week, midday prices should dip. In the cloudy week,
the solar duck curve should be absent.

</details>

<details><summary>Hint 3</summary>

The revenue difference between clear and cloudy weeks shows the
economic value of cloud forecasting.

</details>

<details><summary>Solution</summary>

```python
daytime_csi = features[(features.index.hour >= 8) & (features.index.hour <= 18)]
weekly_csi = daytime_csi["csi"].resample("W").mean().dropna()

clearest_week = weekly_csi.idxmax()
cloudiest_week = weekly_csi.idxmin()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, (label, week_start) in enumerate([
    (f"Clearest (CSI={weekly_csi.max():.2f})", clearest_week - pd.Timedelta(days=6)),
    (f"Cloudiest (CSI={weekly_csi.min():.2f})", cloudiest_week - pd.Timedelta(days=6)),
]):
    week_end = week_start + pd.Timedelta(days=7)
    wk = features.loc[week_start:week_end].dropna(subset=["price"])

    axes[0, col].plot(wk.index, wk.get("solar_gen", wk.get("ghi", pd.Series())),
                       linewidth=1, color="C1")
    axes[0, col].set_ylabel("Solar gen / GHI")
    axes[0, col].set_title(label)

    axes[1, col].plot(wk.index, wk["price"], linewidth=0.8, color="k")
    axes[1, col].axhline(0, color="gray", linewidth=0.5, linestyle="--")
    axes[1, col].set_ylabel("Price ($/MWh)")
    axes[1, col].set_yscale("symlog", linthresh=100)

fig.suptitle(f"{REGION} — clear vs cloudy week", fontsize=13)
fig.tight_layout()
plt.show()
```

Clear weeks show the solar duck curve clearly. Cloudy weeks have
higher midday prices since solar isn't suppressing the market.

</details>

In [ ]:
# Your analysis here

---
## What we learned

1. ERA5 provides hourly gridded weather data useful for renewable
   generation forecasting.
2. The **clear-sky index** isolates cloud effects from the sun's
   position — a seasonally-adjusted measure of solar resource.
3. SA1 is one of the windiest NEM regions, driving its high wind
   generation share.
4. Persistence is a reasonable solar baseline but poor for wind.
5. Both wind and solar have negative price correlations, but at
   different times of day.
6. Cloud forecasting matters: clear vs cloudy weeks produce
   fundamentally different price patterns.

**Next:** Notebook 05 builds the full feature matrix, backtest
harness, baselines, and the dispatch LP.

In [ ]:
# Write report
report_dir = Path(cfg["paths"]["reports"])
report_dir.mkdir(parents=True, exist_ok=True)

report = f"""# Notebook 04 — Weather and Generation Report

Region: {REGION} | Weather: {weather_start} to {weather_end}

## Key findings

- ERA5 data {'loaded from CDS' if has_era5 else 'stubbed (no CDS credentials)'}
- Mean 100m wind speed: {weather['wind_speed_100m'].mean():.1f} m/s
- Wind persistence MAE: {wind_mae:.1f} MW
- Solar persistence MAE: {solar_mae:.1f} MW

## Implemented

- `data.load_era5()` — ERA5 weather download with CDS API
- `features.clear_sky_index()` — actual GHI / clear-sky GHI
"""

(report_dir / "04_weather_generation.md").write_text(report)
print("Report written to", report_dir / "04_weather_generation.md")